# Manufacturing AI Agent — RAG 파트 (v3)

LangGraph 멀티에이전트에서 **RAG 파트만** 분리해, 원본 `manufacturing_agent.ipynb`의 폴더 구조를 따른다.

- **6. `services/rag_service`** : evidence_agent가 호출하는 검색 서비스 (Query Builder ①·Retriever ②·Ranker ③·citation)
- **7. `agents/evidence_agent`** : `state: ManufacturingState`를 받아 서비스 호출 + 근거 요약 → `EvidenceBundle` 반환 (+ 실제 작동 데모)

실행 순서: 0(환경) → 1(LLM) → 2(계약+state) → 4(ChromaDB) → 6(서비스) → 7(에이전트).
사전 준비: `01_embed_documents_chroma.ipynb`로 `document/`를 ChromaDB에 임베딩해 둘 것.


## 0. 설치 & 환경

최초 1회만 실행. 이미 설치돼 있으면 건너뛴다. (uv 권장)


In [12]:
# 최초 1회만 실행 — 주석 해제 후 사용
# !uv pip install langgraph langgraph-checkpoint-sqlite langchain-core chromadb
# (선택) 실제 OpenAI LLM + 임베딩 사용 시 (langchain-openai가 openai 패키지를 함께 설치):
# !uv pip install langchain-openai openai
# (선택) 그래프 시각화:
# !uv pip install grandalf

print("설치 셀: 필요 시 위 주석을 해제해 실행하세요.")

설치 셀: 필요 시 위 주석을 해제해 실행하세요.


In [13]:
from __future__ import annotations

import os
import re
import json
from typing import Any, Optional, Literal

# pydantic (계약 스키마용)
from pydantic import BaseModel, Field

print("기본 import 완료 (RAG 테스트용)")


기본 import 완료 (RAG 테스트용)


## 1. 설정 & LLM 어댑터

`call_llm(system, user)` 하나로 통일한다.
- `langchain-openai` + `OPENAI_API_KEY` 가 있으면 실제 OpenAI 호출
- 없으면 결정론적 **StubLLM** 으로 폴백 → 오프라인에서도 노트북이 끝까지 실행됨


In [14]:
# ===================== 환경설정 (.env 로드) =====================
# API 키는 프로젝트 루트의 .env 파일에서 읽습니다. (.env.example 참고)
# 키를 이 노트북에 직접 적지 마세요 — .env 파일에만 저장합니다 (git에 커밋되지 않음).
# 실행 순서: 먼저 01_embed_documents_chroma.ipynb 를 실행한 뒤 이 노트북을 실행합니다.
#   .env 예시:  OPENAI_API_KEY=sk-proj-XXXXXXXX...

def load_dotenv(path: str = ".env") -> bool:
    if not os.path.exists(path):
        return False
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key and key not in os.environ:
                os.environ[key] = value
    return True

_ENV_PATH = ".env"
_ENV_EXISTS = os.path.exists(_ENV_PATH)
_ENV_LOADED = load_dotenv(_ENV_PATH)

# LangSmith tracing/upload 설정 (.env에서 LANGSMITH_*를 읽음)
LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "")
LANGSMITH_TRACING = os.environ.get("LANGSMITH_TRACING", "true" if LANGSMITH_API_KEY else "false")
LANGSMITH_PROJECT = os.environ.get("LANGSMITH_PROJECT", "manufacturing-agent")
LANGSMITH_ENDPOINT = os.environ.get("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")

os.environ["LANGSMITH_TRACING"] = LANGSMITH_TRACING
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
os.environ["LANGSMITH_ENDPOINT"] = LANGSMITH_ENDPOINT
if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

# LangChain/LangGraph 쪽 호환 환경변수도 같이 맞춘다.
os.environ["LANGCHAIN_TRACING_V2"] = LANGSMITH_TRACING
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT
if LANGSMITH_API_KEY:
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
# =========================================================

# 설정값
DEFAULT_MODEL = os.environ.get("OPENAI_CHAT_MODEL", "gpt-4o")               # 채팅 모델. 비용 민감 시 "gpt-4o-mini"
EMBED_MODEL = os.environ.get("OPENAI_EMBED_MODEL", "text-embedding-3-small") # 임베딩 모델. 고품질은 "text-embedding-3-large"
DATA_DIR = "agent_data"
os.makedirs(DATA_DIR, exist_ok=True)

LONGTERM_DB = os.path.join(DATA_DIR, "longterm_memory.sqlite")   # 장기 메모리 (대화/실행 이력)
CHECKPOINT_DB = os.path.join(DATA_DIR, "checkpoints.sqlite")     # 장기 체크포인터(SqliteSaver)
CHROMA_DIR = os.path.join(DATA_DIR, "chroma")                    # 벡터 스토어

_HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print(".env file:", "OK" if _ENV_EXISTS else "MISSING")
print(".env loaded:", "OK" if _ENV_LOADED else "SKIPPED")
print("OpenAI API key:", "OK" if _HAS_KEY else "MISSING")
print("Chat model:", DEFAULT_MODEL)
print("Embedding model:", EMBED_MODEL)

_LANGSMITH_ENABLED = LANGSMITH_TRACING.lower() in {"1", "true", "yes", "on"}
_LANGSMITH_HAS_KEY = bool(os.environ.get("LANGSMITH_API_KEY"))
print("LangSmith tracing:", "OK" if _LANGSMITH_ENABLED else "OFF")
print("LangSmith API key:", "OK" if _LANGSMITH_HAS_KEY else "MISSING")
print("LangSmith project:", LANGSMITH_PROJECT)
print("LangSmith endpoint:", LANGSMITH_ENDPOINT)

if _LANGSMITH_ENABLED and _LANGSMITH_HAS_KEY:
    try:
        from langsmith import Client
        _ls_client = Client(api_url=LANGSMITH_ENDPOINT, api_key=LANGSMITH_API_KEY)
        next(_ls_client.list_projects(limit=1), None)
        print("LangSmith upload check: OK")
    except Exception as e:
        print("LangSmith upload check: FAILED", e)
else:
    print("LangSmith upload check: SKIPPED")

_llm_client = None
_USE_REAL_LLM = False
try:
    if _HAS_KEY:
        from langchain_openai import ChatOpenAI
        _llm_client = ChatOpenAI(model=DEFAULT_MODEL, temperature=0, max_tokens=1024)
        _USE_REAL_LLM = True
except Exception as e:
    print("실제 LLM 비활성 (StubLLM 사용):", e)


def call_llm(system: str, user: str) -> str:
    """system+user 프롬프트 → 텍스트 응답. 미설치 시 StubLLM 폴백."""
    if _USE_REAL_LLM and _llm_client is not None:
        msg = _llm_client.invoke([("system", system), ("human", user)])
        return msg.content if isinstance(msg.content, str) else str(msg.content)
    return _stub_llm(system, user)


def _stub_llm(system: str, user: str) -> str:
    """결정론적 폴백: 입력을 요약해 자연어처럼 돌려준다(테스트/오프라인용)."""
    head = user.strip().splitlines()[0] if user.strip() else ""
    return f"[stub-llm 요약] {head[:160]}"


print("LLM 모드:", "REAL(" + DEFAULT_MODEL + ")" if _USE_REAL_LLM else "STUB")

.env file: OK
.env loaded: OK
OpenAI API key: OK
Chat model: gpt-4o-mini
Embedding model: text-embedding-3-small
LangSmith tracing: OK
LangSmith API key: OK
LangSmith project: manufacturing-agent
LangSmith endpoint: https://api.smith.langchain.com
LangSmith upload check: OK
LLM 모드: REAL(gpt-4o-mini)


## 2. `contracts/` — 데이터 계약 (Pydantic 스키마)

README 12장. Agent·Gate·Node가 주고받는 구조를 명확한 이름으로 정의한다.
`Artifact` 대신 `PredictionResult` / `EvidenceBundle` / `SafetyDecision` / `FinalAnswer` 등을 쓴다.


In [16]:
# ---------- contracts/context.py ----------
class ConversationTurn(BaseModel):
    role: str
    content: str
    created_at: str

class MachineValue(BaseModel):
    name: str
    value: float | str
    unit: Optional[str] = None
    source: str                       # "current" | "previous"
    is_current: bool
    is_stale: bool = False

class ContextPacket(BaseModel):
    current_question: str
    recent_turns_summary: str = ""
    selected_machine_values: dict[str, MachineValue] = {}
    previous_prediction_result: Optional[PredictionResult] = None
    previous_prediction_summary: Optional[str] = None
    #previous_safety_summary: Optional[str] = None
    user_constraints: dict = {}
    context_warnings: list[str] = []

class AgentContextPacket(BaseModel):
    agent_name: str
    current_question: str
    selected_context: dict = {}
    prior_results: dict = {}

# ---------- contracts/results.py ----------
class PredictionResult(BaseModel):
    status: str
    prediction_label: Optional[str] = None        # "normal" | "failure" 
    failure_types: list[str] = []                 # ["OSF", "TWF"]
    cause_features: list[str] = []                # ["torque", "tool_wear"]
    missing_features: list[str] = []
    full_prediction_available: bool = False
    prediction: Optional[dict] = None             # 계산식 기반 원본 결과
    summary: str = ""

# class EvidenceBundle(BaseModel):
#     retrieval_profile: str     # "default" | "safety" | "manufacturing" | "prediction"
#     queries: list[str] = []    # List of queries used to retrieve evidence
#     documents: list[dict] = [] # List of retrieved documents
#     citations: list[dict] = [] # List of citations for the retrieved documents
#     evidence_summary: str = "" # Summary of the retrieved evidence

class EvidenceBundle(BaseModel):
    mode: str = ""                         # "A" | "B"
    retrieval_profile: str                 # "troubleshooting_rag" | "prediction_plus_rag"
    user_query: str = ""                    # 원본 사용자 질문 (LLM에게는 이걸 그대로 보여줌)
    search_query: str = ""                  # 실제 검색에 사용된 쿼리 (태그/필터링 적용된 형태)
    tags: list[str] = []
    doc_whitelist: Optional[list[str]] = None
    failure_types: list[str] = []
    failure_ko: list[str] = []

    queries: list[str] = []
    documents: list[dict] = []
    citations: list[dict] = []
    evidence_summary: str = ""
    is_prediction_based: bool = False     # 질의가 PredictionResult 기반인지 여부

# class SafetyDecision(BaseModel):
#     risk_level: str                   # none | low | medium | high | critical
#     blocked: bool = False
#     forbidden_actions: list[str] = []
#     required_safety_notes: list[str] = []
#     summary: str = ""

class FinalAnswer(BaseModel):
    answer: str
    citations: list[dict] = []
    warnings: list[str] = []
    missing_inputs: list[str] = []

# ---------- contracts/routing.py ----------
class InputFlags(BaseModel):
    possible_manufacturing_query: bool = False #-> manufacturing 관련 evidence 검색 가능성 -??
    possible_prediction_query: bool = False    #-> prediction 관련 evidence 검색 가능성
    possible_evidence_query: bool = False      #-> RAG evidence 검색 가능성
    possible_prediction_based_evidence_query: bool = False # 추가-> PredictionResult 기반 RAG evidence 검색 가능성
    #possible_safety_query: bool = False
    possible_prompt_injection: bool = False
    contains_sensor_values: bool = False
    blocked_by_raw_input: bool = False

class RouteDecision(BaseModel):
    next_node: str
    reason: str
    stop: bool = False

class GateReport(BaseModel):
    gate_name: str
    status: str
    route_hint: Optional[str] = None
    reason: str = ""
    details: dict = {}

class RunTrace(BaseModel):
    request_id: str
    events: list[dict] = []

print("contracts 정의 완료")

contracts 정의 완료


### 2.1 `contracts/state.py` — LangGraph State

원본 폴더 구조의 그래프 State. v3는 RAG 테스트용이라 `evidence_agent`가 읽는 필드 위주로 둔다.
(`SafetyDecision`은 이 노트북에서 미사용이므로 `safety_decision` 필드는 제외)


In [17]:
# ---------- contracts/state.py ----------
from typing_extensions import TypedDict


class ManufacturingState(TypedDict, total=False):
    # 식별자
    request_id: str
    user_id: str
    thread_id: str
    user_message: str

    # 게이트/라우팅
    input_flags: Optional[InputFlags]
    route: Optional[RouteDecision]
    intent: Optional[str]

    # 컨텍스트
    context_packet: Optional[ContextPacket]
    agent_contexts: dict            # {agent_name: AgentContextPacket}

    # Agent 결과
    prediction_result: Optional[PredictionResult]
    evidence_bundle: Optional[EvidenceBundle]
    final_answer: Optional[FinalAnswer]

    # 검증/재시도/관측
    gate_reports: list
    retry_counts: dict
    run_trace: Optional[RunTrace]


print("ManufacturingState 정의 완료")


ManufacturingState 정의 완료


## 4. ChromaDB RAG 구성

이 노트북은 **2번 실행 노트북**이다. 이미 임베딩된 ChromaDB 컬렉션을 열고 `EvidenceAgent`가 검색만 수행한다.

문서 임베딩은 **1번 준비 노트북**인 `01_embed_documents_chroma.ipynb`에서 최초 1회 또는 문서 변경 시 실행한다.

ChromaDB를 고정 사용한다. 인메모리 키워드 fallback은 두지 않는다.


In [18]:
# ---------- 2) Evidence RAG 런타임: 임베딩된 ChromaDB 검색만 수행 ----------
import hashlib

import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from chromadb.utils import embedding_functions

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180
LOCAL_EMBED_DIM = 384


class LocalHashEmbeddingFunction(EmbeddingFunction[Documents]):
    """외부 모델 다운로드 없이 동작하는 로컬 임베딩 함수."""

    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            vec = [0.0] * LOCAL_EMBED_DIM
            tokens = re.findall(r"[A-Za-z가-힣0-9_]+", text.lower())
            for token in tokens:
                digest = hashlib.sha256(token.encode("utf-8")).digest()
                idx = int.from_bytes(digest[:4], "little") % LOCAL_EMBED_DIM
                sign = 1.0 if digest[4] % 2 == 0 else -1.0
                vec[idx] += sign
            norm = sum(v * v for v in vec) ** 0.5 or 1.0
            vectors.append([v / norm for v in vec])
        return vectors


def build_embedding_function():
    """01_embed_documents_chroma.ipynb의 임베딩 함수와 동일해야 한다."""
    if _HAS_KEY:
        return embedding_functions.OpenAIEmbeddingFunction(
            api_key=os.environ["OPENAI_API_KEY"], model_name=EMBED_MODEL), "manufacturing_document_chunks_openai", f"OpenAI({EMBED_MODEL})"
    return LocalHashEmbeddingFunction(), "manufacturing_document_chunks_local_hash", f"LocalHash({LOCAL_EMBED_DIM})"


_embed_fn, _collection_name, _embed_label = build_embedding_function()
_chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    _chroma_collection = _chroma_client.get_collection(
        _collection_name, embedding_function=_embed_fn)
except Exception as e:
    raise RuntimeError(
        f"ChromaDB 컬렉션 '{_collection_name}'을 찾을 수 없습니다. "
        "먼저 01_embed_documents_chroma.ipynb를 실행해 document/를 임베딩하세요."
    ) from e

print(f"Evidence RAG ChromaDB 연결 완료: collection={_collection_name}, embedding={_embed_label}, chunks={_chroma_collection.count()}")


def vector_search(query: str, k: int = 3, type_filter: Optional[str] = None) -> list[dict]:
    """이미 임베딩된 ChromaDB 컬렉션에서 관련 문서 top-k 검색."""
    where = {"type": type_filter} if type_filter else None
    res = _chroma_collection.query(query_texts=[query], n_results=k, where=where)
    docs = res.get("documents", [[]])[0]
    ids = res.get("ids", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0] if res.get("distances") else [0.0] * len(docs) # 거리 계산 방법: 1 - cosine similarity
    out = []
    for i, doc in enumerate(docs):
        meta = metas[i] or {}
        out.append({
            "id": ids[i],
            "text": doc,
            "type": meta.get("type"),
            "source": meta.get("source"),
            "chunk_index": meta.get("chunk_index"),
            "score": 1.0 - float(distances[i]),
        })
    return out


print("Evidence RAG vector_search 준비 완료")

Evidence RAG ChromaDB 연결 완료: collection=manufacturing_document_chunks_openai, embedding=OpenAI(text-embedding-3-small), chunks=213
Evidence RAG vector_search 준비 완료


## 6. `services/` — RAG 서비스 (evidence_agent가 호출)

`evidence_agent`가 호출하는 검색 로직. 핵심 판단(질의 재작성·검색·랭킹·인용)은 모두 여기서 한다.

- **매핑 레이어**: `FAILURE_RAG_MAP`(고장유형→태그/문서), `VARIABLE_TAG_MAP`(원인변수→태그) + 코드/피처 브리지
- **(1) Query Builder**: profile/예측에 따라 mode A(질의 그대로) / mode B(매핑 태그 재작성 + 문서 화이트리스트)
- **(2) Retriever**: Chroma 검색 (Haas 한정: `type=troubleshooting` + `source=haas/`)
- **(3) Evidence Ranker**: 중복 제거 후 score Top-k
- **citation_service**: 문서 → `{source_id, type, snippet, score}`

진입점: `rag_search(question, profile, prediction)` → `{plan, documents, citations}`


In [19]:
# ---------- services/rag_service.py ----------
# profile -> ChromaDB type 필터 (Haas 문서는 모두 troubleshooting)
## 프로파일이 정의되어야하는 이유 : 각 프로파일에 따라 다른 검색 전략과 문서 필터링을 적용하기 위해
RETRIEVAL_PROFILES = {
    "troubleshooting_rag": "troubleshooting", #mode A: 단순 검색
    "prediction_plus_rag": "troubleshooting", #mode B: 예측 기반 검색
}
HAAS_SOURCE_PREFIX = "haas/"

# ----- 매핑 레이어 -----
# 고장 유형(한글) -> 검색 태그 + 문서 화이트리스트
FAILURE_RAG_MAP = {
    "공구마모고장": {"query_tags": ["tool wear", "chatter", "surface finish", "cutting load",
                              "tool condition", "tool vibration"],
                "documents": ["Mill Chatter", "Mill Spindle"]},
    "열방출실패": {"query_tags": ["overheating", "spindle temperature", "lubrication", "cooling",
                             "air pressure", "thermal issue"],
               "documents": ["Mill Spindle", "Vector Drive"]},
    "과부하파손": {"query_tags": ["overload", "high torque", "cutting force", "spindle load",
                             "chatter", "rpm", "feed speed"],
               "documents": ["Mill Chatter", "Mill Spindle", "Vector Drive"]},
    "전력실패": {"query_tags": ["vector drive", "DC bus", "input voltage", "regen",
                            "electrical failure", "spindle motor", "drive alarm"],
              "documents": ["Vector Drive NGC", "Vector Drive CHC"]},
    "무작위오류": {"query_tags": ["unknown failure", "alarm code", "symptom clarification"],
               "documents": []},
}

#질의에 아래 키워드가 포함되면 이 태그를 추가해 검색 범위를 넓힌다
VARIABLE_TAG_MAP = {
    "기온": ["ambient temperature", "cooling", "overheating"],                 
    "공정온도": ["process temperature", "spindle overheating", "thermal issue", "lubrication"],
    "회전속도": ["rpm", "spindle speed", "chatter", "vibration"],
    "토크": ["torque", "high load", "overload", "cutting force", "spindle load"],
    "공구마모": ["tool wear", "tool condition", "surface finish", "cutting load", "chatter"],
}
# PredictionResult.failure_types의 AI4I 코드 <-> 매핑 테이블(한글 키) 브리지
FAILURE_CODE_TO_KO = {"TWF": "공구마모고장", "HDF": "열방출실패",
                      "OSF": "과부하파손", "PWF": "전력실패", "RNF": "무작위오류"}
FEATURE_TO_KO = {"air_temperature": "기온", "process_temperature": "공정온도",
                 "rotational_speed": "회전속도", "torque": "토크", "tool_wear": "공구마모"}


#(1) Query Builder------------------------------
def build_query(question: str, profile: str, prediction: Optional[PredictionResult] = None) -> dict:
    """
    Query Builder: 사용자 질문과 Prediction 결과를 기반으로 RAG 검색 계획(Search Plan)을 생성한다.

    Mode A (단순 문서 검색)
        - prediction 정보가 없거나 profile이 troubleshooting_rag인 경우
        - 사용자 질의를 그대로 검색 Query로 사용한다.

    Mode B (예측 기반 문서 검색)
        - Prediction 결과의 고장 유형(failure_types)과
          원인 변수(cause_features)를 이용하여
          검색 태그와 문서 화이트리스트를 생성한다.

    Args:
        question: 사용자의 원본 질문.
        profile: Retrieval Profile. ("troubleshooting_rag", "prediction_plus_rag")
        prediction: Prediction Agent 결과.Mode B에서만 사용된다.

    Returns:
        Search Plan(dict)
            {
                mode,
                profile,
                user_query,
                search_query,
                tags,
                doc_whitelist,
                failure_types,
                failure_ko
            }
    """

    has_pred = bool(prediction and prediction.failure_types) #
    if profile != "prediction_plus_rag" or not has_pred:
        return {"mode": "A", "profile": profile, "user_query": question, "search_query": question,
                "tags": [], "doc_whitelist": None, "failure_types": [], "failure_ko": []}

    # mode B: 도출된 고장 유형을 모두 반영 + 원인 변수 태그 확장
    failure_types, failure_ko, tags, docs = [], [], [], []
    for code in prediction.failure_types:
        failure_types.append(code)
        ko = FAILURE_CODE_TO_KO.get(code)
        failure_ko.append(ko)
        fmap = FAILURE_RAG_MAP.get(ko, {})
        tags.extend(fmap.get("query_tags", []))
        docs.extend(fmap.get("documents", []))
    for feat in (prediction.cause_features or []):
        tags.extend(VARIABLE_TAG_MAP.get(FEATURE_TO_KO.get(feat, ""), []))
    tags = list(dict.fromkeys(tags))
    docs = list(dict.fromkeys(docs))
    return {"mode": "B", "profile": profile, "user_query": question,
            "search_query": " ".join([question, *tags]).strip(), "tags": tags,
            "doc_whitelist": docs or None, "failure_types": failure_types, "failure_ko": failure_ko}


def _doc_name_matches(source: str, doc_name: str) -> bool:
    """
    화이트리스트 문서명과 실제 source 경로가 일치하는지 확인한다.

    문서명을 공백 기준으로 분리한 뒤,
    모든 토큰이 source 경로에 포함되는지 검사한다.

    Args:
        source:
            검색 결과의 source 경로.

        doc_name:
            화이트리스트에 등록된 문서명.

    Returns:
        True이면 해당 문서로 인정,
        False이면 제외한다.
    """
    s = (source or "").lower()
    return all(tok.lower() in s for tok in doc_name.split())

#(2) Retriever------------------------------
def retrieve_stage(plan: dict, k: int = 8) -> list[dict]:
    """
    Retriever.
    Query Builder가 생성한 Search Plan을 이용하여 ChromaDB에서 문서를 검색한다.

    수행 과정
        1. Retrieval Profile에 맞는 type filter 적용
        2. Vector Search 수행
        3. Haas 문서만 필터링
        4. (Mode B인 경우) 문서 화이트리스트 적용

    Args:
        plan:
            build_query()가 생성한 Search Plan.

        k:
            Vector Search 후보 문서 개수.

    Returns:
        검색된 문서 후보 리스트.
    """
    type_filter = RETRIEVAL_PROFILES.get(plan["profile"], "troubleshooting")
    hits = vector_search(plan["search_query"], k=k, type_filter=type_filter)
    hits = [h for h in hits if (h.get("source") or "").startswith(HAAS_SOURCE_PREFIX)]
    whitelist = plan.get("doc_whitelist")
    if whitelist:
        hits = [h for h in hits
                if any(_doc_name_matches(h.get("source", ""), n) for n in whitelist)]
    return hits


#(3) Evidence Ranker------------------------------
def rank_evidence(hits: list[dict], top_k: int = 3) -> list[dict]:
    """
    Evidence Ranker.
    Retriever가 반환한 후보 문서를 정렬하고 중복 Chunk를 제거하여 최종 근거 문서를 선택한다.

    수행 과정
        1. score 기준 정렬
        2. (source, chunk_index) 기준 중복 제거
        3. Top-k 문서 선택

    Args:
        hits:
            Retriever 검색 결과.

        top_k:
            최종 선택할 근거 문서 개수.

    Returns:
        최종 근거 문서 리스트.
    """
    seen, ranked = set(), []
    for h in sorted(hits, key=lambda x: x.get("score", 0.0), reverse=True):
        key = (h.get("source"), h.get("chunk_index"))
        if key in seen:
            continue
        seen.add(key)
        ranked.append(h)
        if len(ranked) >= top_k:
            break
    return ranked


#(4) Citation Builder------------------------------
def build_citations(docs: list[dict]) -> list[dict]:
    """
    Citation Builder.

    최종 선택된 문서를 Citation 형태로 변환한다.

    각 Citation에는
        - source_id
        - document type
        - snippet
        - retrieval score

    를 포함한다.

    Args:
        docs:
            최종 근거 문서 리스트.

    Returns:
        Citation 정보 리스트.
    """
    return [{"source_id": d["id"], "type": d.get("type"),
             "snippet": d["text"][:120], "score": round(float(d.get("score", 0)), 3)}
            for d in docs]


#----------------- RAG Search Pipeline (Entry Point) ------------------------------
def rag_search(question: str, profile: str, prediction: Optional[PredictionResult] = None,
               retrieve_k: int = 8, top_k: int = 3) -> dict:
    """
    RAG Search Pipeline.

    Evidence Agent가 호출하는 RAG 서비스의 진입점이다.

    내부 수행 순서
        1. Query Builder
        2. Retriever
        3. Evidence Ranker
        4. Citation Builder

    Note:
        문서 요약 및 자연어 답변 생성은 수행하지 않는다.
        Evidence Agent가 반환된 documents와 citations를
        이용하여 최종 답변을 생성한다.

    Args:
        question:
            사용자 질문.

        profile:
            Retrieval Profile.

        prediction:
            Prediction Agent 결과.
            Mode B에서만 사용된다.

        retrieve_k:
            Retriever 후보 문서 개수.

        top_k:
            최종 근거 문서 개수.

    Returns:
        {
            "plan": Search Plan,
            "documents": Ranked Documents,
            "citations": Citation List
        }
    """
    plan = build_query(question, profile, prediction)   # (1)
    hits = retrieve_stage(plan, k=retrieve_k)            # (2)
    ranked = rank_evidence(hits, top_k=top_k)            # (3)
    return {"plan": plan, "documents": ranked, "citations": build_citations(ranked)}


print("rag_service / citation_service 정의 완료")


rag_service / citation_service 정의 완료


## 7. `agents/` — evidence_agent (실제 작동)

SubAgent는 독립 판단만 한다. 입력은 `ManufacturingState`(원본과 동일 시그니처),
핵심 검색은 6번 `rag_search`에 위임하고, LLM은 **근거 요약**에만 보조로 쓴 뒤 `EvidenceBundle`을 만든다.


In [20]:
# ---------- agents/evidence_agent/agent.py ----------
EVIDENCE_SUMMARY_SYSTEM = (
    "너는 근거 수집가다. 검색 문서를 바탕으로 핵심 근거를 2~3문장으로 요약하라. "
    "문서에 없는 내용은 만들지 마라."
)


def _pick_profile(pred: Optional[PredictionResult]) -> str:
    """예측에 고장 유형이 있으면 예측 결합 검색, 아니면 일반 트러블슈팅 검색."""
    if pred and pred.failure_types:
        return "prediction_plus_rag"
    return "troubleshooting_rag"


def evidence_agent(state: ManufacturingState) -> dict:
    ctx = state["agent_contexts"]["evidence_agent"]
    pred = state.get("prediction_result")
    profile = _pick_profile(pred)

    # RAG 서비스 호출
    result = rag_search(ctx.current_question, profile, pred)   # 6번 RAG 서비스 호출
    plan, docs, citations = result["plan"], result["documents"], result["citations"]

    # 문서 요약 (LLM 호출) 
    summary = call_llm(
        EVIDENCE_SUMMARY_SYSTEM,
        f"질문:{ctx.current_question}\n문서:{json.dumps([d['text'] for d in docs], ensure_ascii=False)}")

    # EvidenceBundle 생성
    bundle = EvidenceBundle(
        mode=plan["mode"],
        retrieval_profile=plan["profile"],
        user_query=plan["user_query"],
        search_query=plan["search_query"],
        tags=plan["tags"],
        doc_whitelist=plan["doc_whitelist"],
        failure_types=plan["failure_types"],
        failure_ko=plan["failure_ko"],
        queries=[plan["search_query"]],
        documents=docs,
        citations=citations,
        evidence_summary=summary,
        is_prediction_based=(plan["mode"] == "B"),
    )
    return {"evidence_bundle": bundle}


print("evidence_agent 정의 완료")



evidence_agent 정의 완료


In [21]:

# ===================== 실제 에이전트 작동 =====================
def _print_bundle(tag: str, out: dict) -> None:
    b = out["evidence_bundle"]
    print(f"\n[{tag}] mode={b.mode} profile={b.retrieval_profile}")
    if b.failure_types:
        print("  failure_types:", list(zip(b.failure_types, b.failure_ko)))
    print("  search_query:", b.search_query[:80])
    if b.doc_whitelist:
        print("  doc_whitelist:", b.doc_whitelist)
    print(f"  documents={len(b.documents)} citations={len(b.citations)}")
    print("  evidence_summary:", b.evidence_summary)
    for c in b.citations:
        print(f"    - {c['source_id']} ({c['type']}, score={c['score']})")


# 시나리오 1) mode A — 예측 없음 (단순 질의)
_state_a = {"agent_contexts": {"evidence_agent": AgentContextPacket(
    agent_name="evidence_agent", current_question="밀링 채터(chatter)가 발생하는 원인과 해결 방법은?")}}
_print_bundle("mode A", evidence_agent(_state_a))




[mode A] mode=A profile=troubleshooting_rag
  search_query: 밀링 채터(chatter)가 발생하는 원인과 해결 방법은?
  documents=3 citations=3
  evidence_summary: 밀링 채터는 주로 너무 많은 플루트가 절삭에 참여하거나, 공구 경로의 변화로 인해 절삭력이 급증할 때 발생한다. 이를 해결하기 위해서는 플루트 수를 줄이거나 절삭 깊이 및 폭을 감소시키고, 일정한 절삭력을 유지하는 공구 경로를 사용하는 것이 효과적이다. 또한, 공구의 마모 상태나 길이, 칩 하중, 쿨런트 문제 등도 점검해야 한다.
    - b0088134a96305e1 (troubleshooting, score=0.425)
    - 242f2c9764ba8c39 (troubleshooting, score=0.374)
    - 63c489bd31879be5 (troubleshooting, score=0.348)


In [22]:

# 시나리오 2) mode B — PredictionResult(고장 유형 + 원인 변수) 주입
_pred = PredictionResult(
    status="PARTIAL",
    failure_types=["OSF", "TWF"],
    cause_features=["torque", "tool_wear", "rotational_speed"],
)
_state_b = {"agent_contexts": {"evidence_agent": AgentContextPacket(
    agent_name="evidence_agent", current_question="스핀들 부하가 높을 때 점검해야 할 항목은?")},
    "prediction_result": _pred}
_print_bundle("mode B", evidence_agent(_state_b))


[mode B] mode=B profile=prediction_plus_rag
  failure_types: [('OSF', '과부하파손'), ('TWF', '공구마모고장')]
  search_query: 스핀들 부하가 높을 때 점검해야 할 항목은? overload high torque cutting force spindle load chatter
  doc_whitelist: ['Mill Chatter', 'Mill Spindle', 'Vector Drive']
  documents=3 citations=3
  evidence_summary: 스핀들 부하가 높을 때 점검해야 할 항목으로는 스핀들 윤활 시스템의 작동 여부, 도구 홀더 및 스핀들 테이퍼의 청소 및 손상 여부, 그리고 베어링 상태를 확인하는 것이 중요하다. 또한, 도구의 길이가 너무 길거나 깊이 절삭이 과도한 경우에도 부하가 증가할 수 있으므로, 도구 길이를 줄이거나 절삭 깊이를 조정하는 것이 필요하다.
    - 084cdf527f87df34 (troubleshooting, score=0.556)
    - 59310f70c706b4e6 (troubleshooting, score=0.555)
    - 607ec1b2d3c3edda (troubleshooting, score=0.553)


### evidence gate는 생략

### final answer node

In [ ]:
# ---------- nodes/final_answer_node.py ----------
def final_answer_node(state: ManufacturingState) -> dict:
    pred = state.get("prediction_result")
    ev = state.get("evidence_bundle")
    # sd = state.get("safety_decision")   # safety 측면 보류
    packet = state.get("context_packet")

    warnings: list[str] = list(packet.context_warnings) if packet else []
    missing = pred.missing_features if pred else []
    citations = ev.citations if ev else []

    parts = []
    if pred:
        if pred.full_prediction_available:
            parts.append(f"[예측] {pred.summary}")
        elif pred.partial_risks:
            risk_str = ", ".join(f"{r.failure_type}={r.level}({r.score})" for r in pred.partial_risks)
            parts.append(f"[부분 예측] {risk_str}. {pred.summary}")
            if missing:
                parts.append(f"전체 예측은 누락값 때문에 불가: {missing}")
        else:
            parts.append(f"[예측 불가] 필요한 입력이 부족합니다: {missing}")
        if pred.used_stale_features:
            parts.append(f"[맥락] 이전 턴 값 사용: {pred.used_stale_features}")
        if pred.limitations:
            parts.append("[제약] " + " ".join(pred.limitations))
    if ev and ev.evidence_summary:
        parts.append(f"[근거] {ev.evidence_summary}")
    # if sd:
    #     if sd.blocked:
    #         parts.append("[안전] 해당 요청은 안전 정책상 수행할 수 없습니다.")
    #     if sd.required_safety_notes:
    #         parts.append("[안전 권고] " + " ".join(sd.required_safety_notes))

    answer = "\n".join(parts) if parts else "현재 입력만으로는 판단할 수 있는 내용이 없습니다."
    fa = FinalAnswer(answer=answer, citations=citations, warnings=warnings, missing_inputs=missing)
    return {"final_answer": fa}
print("final_answer_node 정의 완료")